# Inteligência Artificial (PCO118A) — Aula 8: Busca Local
**Prof. Carlos Costa Cedro** · Bacharelado em Ciência da Computação · PUCPR

Complete as **seis lacunas** marcadas com `TODO`. Execute as células na ordem.

O problema é o das **oito rainhas**: posicionar oito rainhas em um tabuleiro 8×8 sem que
nenhuma ataque outra.

A espinha dorsal deste laboratório é o **contador de avaliações da função objetivo**.
Ele é instrumentado na lacuna 1 e usado até o fim. É ele que torna possível a comparação
justa da lacuna 6, e é o hábito que o TDE3 vai cobrar.

> **Sobre reprodutibilidade.** Os métodos desta aula são estocásticos: duas execuções dão
> resultados diferentes. As verificações automáticas usam **faixas**, e não valores exatos.
> Um laboratório que exigisse igualdade exata de um valor estocástico estaria ensinando a
> coisa errada. Anote sempre a semente que você usou.

In [ ]:
# ============================================================
# CÓDIGO FORNECIDO — não altere
# ============================================================
import random, math, time

N = 8

# Contador global de avaliações da função objetivo.
# TODA chamada a conflitos() deve incrementá-lo.
AVALIACOES = 0

def zerar():
    global AVALIACOES
    AVALIACOES = 0

def mostrar(s):
    """s[c] = linha da rainha na coluna c."""
    for r in range(N):
        print(' '.join('Q' if s[c] == r else '.' for c in range(N)))

print('Setup carregado.')
mostrar([0, 4, 7, 5, 2, 6, 1, 3])

## Lacuna 1 — representação e função objetivo

**Representação A, vetor livre.** O estado é uma lista de 8 posições; `s[c]` é a linha da
rainha da coluna c. Como há exatamente uma rainha por coluna, conflitos de coluna são
impossíveis por construção.

Implemente `conflitos(s)`: o número de **pares** de rainhas que se atacam. Duas rainhas nas
colunas i e j se atacam se estão na mesma linha (`s[i] == s[j]`) ou na mesma diagonal
(`abs(s[i] - s[j]) == j - i`).

Não esqueça de incrementar `AVALIACOES`.

In [ ]:
def conflitos(s):
    global AVALIACOES
    # TODO: incrementar AVALIACOES e contar os pares que se atacam
    raise NotImplementedError('lacuna 1')

In [ ]:
# Testes de sanidade (fornecidos)
assert conflitos([0, 4, 7, 5, 2, 6, 1, 3]) == 0, 'esta é uma solução conhecida'
assert conflitos([0, 1, 2, 3, 4, 5, 6, 7]) == 28, 'diagonal principal: todos os pares se atacam'
assert conflitos([0, 0, 0, 0, 0, 0, 0, 0]) == 28, 'todas na mesma linha'
assert conflitos([0, 2, 4, 6, 1, 3, 5, 7]) == 1, 'quase uma solução: resta um par em diagonal'
assert conflitos([0, 5, 7, 2, 6, 3, 1, 4]) == 0, 'outra das 92 soluções'

print(f'espaço de estados da representação A: {N**N} = {N}^{N}')
print('Lacuna 1: OK')

## Lacuna 2 — Hill Climbing por subida mais íngreme

O problema é de **minimização**: queremos zero conflitos. O algoritmo é o da seção 3.3.1
do material teórico, com o sinal invertido.

Vizinhança: mover **uma** rainha para outra casa da **mesma coluna**. São 8 × 7 = 56 vizinhos.

`hill_climbing(s)` devolve `(conflitos_finais, passos)`. Em caso de empate entre vizinhos
igualmente bons, sorteie um deles.

> **Atenção.** Só aceite melhora **estrita**. Aceitar empate faz o algoritmo passear
> indefinidamente em platôs.

In [ ]:
def hill_climbing(s):
    s = list(s)
    passos = 0
    while True:
        # TODO: avaliar os 56 vizinhos, achar o melhor valor;
        #       se nenhum for estritamente melhor que o atual, devolver (conflitos(s), passos);
        #       senão sortear entre os melhores, mover e contar o passo
        raise NotImplementedError('lacuna 2')

In [ ]:
def estado_vetor():
    return [random.randrange(N) for _ in range(N)]

def experimento(hc, gerador, n=1000, semente=42):
    random.seed(semente)
    sucessos, p_ok, p_fail = 0, [], []
    for _ in range(n):
        h, p = hc(gerador())
        if h == 0: sucessos += 1; p_ok.append(p)
        else:                     p_fail.append(p)
    return (sucessos / n,
            sum(p_ok) / max(len(p_ok), 1),
            sum(p_fail) / max(len(p_fail), 1))

t = time.time()
taxa_A, pok_A, pfa_A = experimento(hill_climbing, estado_vetor)
print(f'representação A (vetor), 1000 instâncias, semente 42')
print(f'  taxa de sucesso       = {100*taxa_A:.1f}%')
print(f'  passos médios sucesso = {pok_A:.1f}')
print(f'  passos médios falha   = {pfa_A:.1f}')
print(f'  tempo = {time.time()-t:.1f} s')

assert 0.10 <= taxa_A <= 0.18, 'esperado algo em torno de 13% a 14%'
print('\nLacuna 2: OK')

### Para o relatório

O algoritmo **falha em cerca de 86% dos casos**, e falha em pouco mais de 3 passos.
Um espaço de quase 17 milhões de estados é atravessado em três ou quatro movimentos.

**Isso é defeito ou é a assinatura do método?** Justifique em uma frase.

## Lacuna 3 — trocar a representação

**Representação B, permutação.** O estado é uma permutação de 0 a 7: `s[c]` continua sendo
a linha da rainha da coluna c, mas nenhuma linha se repete. Conflitos de linha passam a ser
impossíveis **por construção**, e só sobram os diagonais.

Vizinhança: **trocar** as linhas de duas colunas. São C(8,2) = 28 vizinhos.

A função objetivo é a **mesma**: `conflitos(s)` não muda. O algoritmo é o **mesmo**: subida
mais íngreme. Só a vizinhança muda.

In [ ]:
def hill_climbing_perm(s):
    s = list(s)
    passos = 0
    while True:
        # TODO: mesma estrutura da lacuna 2, mas a vizinhança é trocar s[i] com s[j]
        raise NotImplementedError('lacuna 3')

def estado_perm():
    return random.sample(range(N), N)

In [ ]:
taxa_B, pok_B, pfa_B = experimento(hill_climbing_perm, estado_perm)

print(f"{'':<26}{'A (vetor)':>12}{'B (permutação)':>17}")
print(f"{'tamanho do espaço':<26}{N**N:>12}{math.factorial(N):>17}")
print(f"{'vizinhos por estado':<26}{N*(N-1):>12}{math.comb(N,2):>17}")
print(f"{'taxa de sucesso':<26}{100*taxa_A:>11.1f}%{100*taxa_B:>16.1f}%")
print(f"{'passos médios (sucesso)':<26}{pok_A:>12.1f}{pok_B:>17.1f}")
print(f"{'passos médios (falha)':<26}{pfa_A:>12.1f}{pfa_B:>17.1f}")

assert taxa_B > 2 * taxa_A, 'a representação B deve ser muito melhor'
print('\nLacuna 3: OK')

### Para o relatório

A taxa de sucesso salta sem que **uma linha do algoritmo** mude. **Explique o mecanismo**,
e não apenas o fato: que tipo de conflito a representação B torna impossível por
construção, e o que isso faz com a paisagem que o algoritmo percorre?

## Lacuna 4 — reinício aleatório e o Teorema 2

O Teorema 2 diz que, se cada execução independente tem probabilidade p de sucesso, o número
esperado de execuções até o primeiro sucesso é **1/p**.

Implemente `reinicio_aleatorio()`, que executa Hill Climbing repetidamente a partir de
estados sorteados até obter zero conflitos, devolvendo quantas execuções foram necessárias.
Depois compare a média observada com a previsão do teorema.

In [ ]:
def reinicio_aleatorio(hc, gerador):
    # TODO: repetir hc(gerador()) até obter 0 conflitos; devolver o número de execuções
    raise NotImplementedError('lacuna 4')

In [ ]:
random.seed(7)
M = 2000
tentativas = [reinicio_aleatorio(hill_climbing, estado_vetor) for _ in range(M)]
media = sum(tentativas) / M

print(f'reinícios médios até o sucesso (medido, {M} execuções) = {media:.2f}')
print(f'p implicado por 1/média                                = {1/media:.4f}')
print(f'p medido na lacuna 2                                   = {taxa_A:.4f}')
print(f'previsão do Teorema 2 com o p da lacuna 2 (1/p)        = {1/taxa_A:.2f}')
print(f'maior número de reinícios observado                    = {max(tentativas)}')

assert abs(1/media - taxa_A) < 0.04, 'previsão e medição devem concordar'
print('\nLacuna 4: OK')

### Para o relatório

1. O p implicado pela média de reinícios e o p medido na lacuna 2 não são idênticos.
   **Isso invalida o Teorema 2?** Justifique em termos de variação amostral.
2. Olhe o maior número de reinícios observado. A média é baixa, mas a **cauda** é longa.
   O que isso significa para quem precisa de um tempo de resposta garantido?

## Lacuna 5 — Simulated Annealing

Implemente o algoritmo da seção 3.4.2, com **resfriamento geométrico**: a cada iteração,
`T = T * alfa`.

Como o problema é de minimização, o movimento **melhora** quando reduz os conflitos.
Definindo `d = conflitos_atual - conflitos_novo`, temos `d > 0` para melhora, e a
probabilidade de aceitar uma piora é `exp(d / T)`.

Vizinho: sorteie uma coluna e uma linha. **Sorteie**, não escolha o melhor: é isso que
torna o Simulated Annealing aplicável a vizinhanças grandes demais para enumerar.

In [ ]:
def simulated_annealing(s, T0=2.0, alfa=0.99, Tmin=1e-3):
    s = list(s)
    e = conflitos(s)
    T = T0
    it = 0
    while T > Tmin and e > 0:
        it += 1
        # TODO: sortear coluna c e linha r; se r == s[c], apenas resfriar e continuar;
        #       calcular e2 = conflitos do vizinho e d = e - e2;
        #       aceitar se d > 0 ou se random.random() < exp(d/T); senão desfazer;
        #       resfriar: T *= alfa
        raise NotImplementedError('lacuna 5')
    return e, it

In [ ]:
random.seed(11)
print(f"{'alfa':>7}{'iter. até Tmin':>17}{'sucesso':>10}{'iterações médias':>19}")
tabela = {}
for alfa in (0.95, 0.99, 0.999):
    suc, its = 0, []
    for _ in range(1000):
        e, it = simulated_annealing(estado_vetor(), alfa=alfa)
        if e == 0: suc += 1
        its.append(it)
    teorico = math.log(1e-3 / 2.0) / math.log(alfa)
    tabela[alfa] = (suc / 1000, sum(its) / 1000)
    print(f'{alfa:>7}{teorico:>17.0f}{100*suc/1000:>9.1f}%{sum(its)/1000:>19.0f}')

assert tabela[0.999][0] > tabela[0.99][0] > tabela[0.95][0], 'resfriar mais devagar deve ajudar'
assert tabela[0.999][0] > 0.90
print('\nLacuna 5: OK')

### Para o relatório

1. A coluna `iter. até Tmin` é o **pior caso**: quantas iterações o resfriamento levaria se
   nunca encontrasse solução. A coluna `iterações médias` é bem menor. **Por quê?**
2. Passar de α = 0,95 para α = 0,999 multiplica as iterações por cerca de 12 e o sucesso por
   cerca de 6. **O ganho compensa?** A resposta depende de uma informação que não está na
   tabela. Qual?
3. O resfriamento geométrico **não** satisfaz a condição do Teorema 3, e portanto este
   algoritmo **não tem garantia alguma** de encontrar o ótimo. Ainda assim resolve quase
   tudo. Como você justificaria seu uso em um relatório profissional?

## Lacuna 6 — comparação justa

Agora a pergunta que o TDE3 vai fazer: **entre reinício aleatório e Simulated Annealing,
qual é melhor?**

A comparação só é honesta se fixar a mesma unidade de custo, e essa unidade é o
**número de avaliações da função objetivo**. Não é segundos, que dependem da máquina, e
não é passos, que custam coisas diferentes em cada algoritmo.

Meça, para cada método, quantas avaliações de `conflitos` são gastas.

In [ ]:
# TODO: para cada um dos dois métodos, rodar 500 execuções zerando AVALIACOES antes de cada
#       uma, e registrar a média de AVALIACOES e a taxa de sucesso.
#       Para o reinício aleatório, medir até o sucesso (a taxa é 100% por construção).
#       Para o SA, usar alfa=0.999 e medir uma execução de cada vez.
raise NotImplementedError('lacuna 6')

### Para o relatório

1. Os dois métodos ficam próximos em avaliações de f. **Se você tivesse contado passos**,
   qual dos dois pareceria dezenas de vezes melhor, e por que a conclusão seria falsa?
2. Compare o custo dos dois com o tamanho do espaço de estados. Vale a pena usar busca
   local aqui? E em 100 rainhas, cujo espaço tem 100¹⁰⁰ estados?
3. Os dois métodos empatam neste problema. **Que outros critérios** você usaria para
   escolher entre eles em um projeto real?

---
## Entrega

O relatório deve conter:

- as respostas às perguntas das lacunas 2 a 6;
- **número de execuções, semente e distribuição** em toda medição de método estocástico,
  com no mínimo média, melhor e pior resultado. Relatórios que apresentem **apenas a melhor
  execução serão devolvidos**;
- custo sempre em **avaliações da função objetivo**, nunca em segundos;
- a tabela da lacuna 6, que é o embrião do protocolo experimental do **TDE3 (Otimização)**,
  com execução entre 28/10 e 04/11.